### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [138]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.2
[notice] To update, run: c:\Users\Nico\.pyenv\pyenv-win\versions\3.11.8\python.exe -m pip install --upgrade pip


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [139]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [140]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [141]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [142]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [143]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [144]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [145]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [146]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [147]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [148]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [149]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [150]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [151]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [152]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [153]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [154]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [155]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [156]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [157]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [158]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [159]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [160]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

### Alumno: Nicolás Moguilner Reh (a2623)



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## Ejercicio 1: Similaridad de documentos

Tomamos 5 documentos al azar de train (con semilla fija para que sea reproducible).

In [161]:
rng = np.random.default_rng(42)

idxs = rng.choice(X_train.shape[0], size=5, replace=False)
docs = X_train[idxs]
print(idxs)

[8754 4965 7404 1009 4899]


Medimos la similaridad coseno de los 5 documentos con todos los documentos de train. Obtenemos una fila por documento.

In [162]:
cossim_all = cosine_similarity(docs, X_train)
cossim_all.shape

(5, 11314)

Para cada documento obtenemos los 5 más similares (salteando el primero, que es el documento mismo) y mostramos su clase, similaridad y un extracto del texto para analizar si la similaridad tiene sentido.

In [163]:
for idx, cossim in zip(idxs, cossim_all):
  mostsim = np.argsort(cossim)[::-1][1:6]

  print('=' * 80)
  print(f'DOC {idx} | clase: {newsgroups_train.target_names[y_train[idx]]}')
  print(newsgroups_train.data[idx][:300].strip())
  print('-' * 80)

  for i in mostsim:
    print(f'  [{i}] sim={cossim[i]:.3f} | clase: {newsgroups_train.target_names[y_train[i]]}')
    print(f'      {newsgroups_train.data[i][:200].strip()!r}')

  print(f'\n  -> {(y_train[mostsim] == y_train[idx]).sum()}/5 vecinos con la misma clase')

DOC 8754 | clase: talk.religion.misc
/(hudson)
/If someone inflicts pain on themselves, whether they enjoy it or not, they
/are hurting themselves.  They may be permanently damaging their body.

That is true.  It is also none of your business.  

Some people may also reason that by reading the bible and being a Xtian
you are permanent
--------------------------------------------------------------------------------
  [6552] sim=0.490 | clase: talk.religion.misc
      "If I have a habit that I really want to break, and I am willing to\nmake whatever sacrifice I need to make to break it, then I do so.\nThere have been bad habits of mine that I've decided to put forth"
  [10613] sim=0.481 | clase: talk.religion.misc
      "/(hudson)\n/Yes you do.  Who is to say that it is immoral for onesself to experience\n/pain or to be hurt in some other way.  Maybe unpleasant, but that doesn't\n/say anything about morality.\n\nIt violate"
  [3616] sim=0.465 | clase: talk.religion.misc
      'And I main

**Interpretación:** En 3 de los 5 documentos la mayoría de los vecinos comparte la clase, y la similaridad tiene sentido por el contenido:
* **Doc 8754 (talk.religion.misc):** 4/5 vecinos de la misma clase, con similaridades altas (~0.46-0.49). Comparten tema (dolor, hábitos, la Biblia) e incluso el mismo hilo de discusión (el texto citado de `/(hudson)`). El vecino de `talk.politics.mideast` aparece por vocabulario de debate y citas, no por tema.
* **Doc 1009 (talk.politics.guns):** 4/5 de la misma clase (armas, licencias, clubes de tiro). El de `alt.atheism` coincide por la palabra "religion" y por el estilo de citas con `>` y `:`.
* **Doc 4899 (sci.crypt):** 3/5. Es una discusión sobre libertad de información y regulación de la red, así que se parece a otros posts de crypto, pero también a posts polémicos de `alt.atheism` y `talk.politics.mideast` con estilo de flame.
* **Doc 4965 (comp.sys.mac.hardware):** 2/5. Habla de puertos (printer port, modem port). Los vecinos son todos de hardware (`comp.sys.mac.hardware`, `comp.sys.ibm.pc.hardware`, `comp.graphics`), con lo cual la similaridad es coherente con el contenido aunque la etiqueta no coincida: los grupos `comp.*` son temáticamente cercanos.
* **Doc 7404 (comp.os.ms-windows.misc):** 0/5. Es un texto muy corto y genérico ("application", "hello"). Los vecinos coinciden en esas palabras sueltas, incluso con documentos que son solo "Hello," (de `comp.graphics` y `sci.med`), por lo que la similaridad es poco informativa.

Las similaridades son más altas (0.4-0.5) cuando hay vocabulario temático específico y bajas (0.15-0.27) en documentos cortos o con vocabulario muy general. En general, los errores de etiqueta son entre grupos cercanos (`comp.*`, `talk.*`/`alt.atheism`) o por documentos con poco contenido.

## Ejercicio 2: Clasificación por prototipos (tipo zero-shot)

Para cada documento de test calculamos la similaridad coseno contra todos los de train y asignamos la clase del documento más similar (`argmax`).

La matriz completa de similaridades (7532 x 11314) ocupa cerca de 680 MB, por lo que la calculamos por bloques para no saturar la memoria.

In [164]:
batch = 1000
y_pred_proto = np.empty(X_test.shape[0], dtype=y_train.dtype)

for start in range(0, X_test.shape[0], batch):
  sim = cosine_similarity(X_test[start:start + batch], X_train)
  y_pred_proto[start:start + batch] = y_train[sim.argmax(axis=1)]

Evaluamos con el F1-score macro, igual que con Naïve Bayes, para poder comparar ambos modelos.

In [165]:
f1_score(y_test, y_pred_proto, average='macro')

0.5049911553681621

Vemos el desempeño por clase para identificar con qué grupos se confunde más el clasificador.

In [166]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_proto, target_names=newsgroups_test.target_names))

                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33      0.41       393
                 sci.med       0.65      0.49      0.56       396
         

**Interpretación:** El clasificador por similaridad alcanza un F1 macro de 0.505, bastante menor que el 0.585 de Naïve Bayes (sin ajustar). Al decidir con un único documento (el más similar), es muy sensible al ruido: basta que un post corto o genérico sea el más parecido para equivocar la clase.

Por clase, las que mejor funcionan tienen vocabulario muy específico: `rec.sport.hockey` (F1 0.73) y `comp.windows.x` (0.64). Las peores son `talk.religion.misc` (0.28), `talk.politics.misc` (0.31) y `sci.electronics` (0.41), que son grupos con temas amplios y que se solapan con otros (`alt.atheism` y `soc.religion.christian`, `talk.politics.*`). Es lo mismo que observamos en el ejercicio 1: las confusiones se dan entre temas cercanos. Además, este método es costoso en inferencia porque compara cada documento contra todo el train.

## Ejercicio 3: Modelos Naïve Bayes

Probamos distintas configuraciones del vectorizador (TF-IDF o conteos, con y sin stop words, con distinto `min_df`) y de los modelos (`MultinomialNB` y `ComplementNB`, con distintos valores de `alpha`). No modificamos `ngram_range`.

Evaluamos todas las combinaciones con el F1-score macro sobre el conjunto de test.

In [167]:
resultados = []

for vect_name, Vect in [('tfidf', TfidfVectorizer), ('count', CountVectorizer)]:
  for stop_words in [None, 'english']:
    for min_df in [1, 2]:
      vect = Vect(stop_words=stop_words, min_df=min_df)
      X_tr = vect.fit_transform(newsgroups_train.data)
      X_te = vect.transform(newsgroups_test.data)

      for model_name, Model in [('MultinomialNB', MultinomialNB), ('ComplementNB', ComplementNB)]:
        for alpha in [0.01, 0.03, 0.1, 0.3, 1.0]:
          model = Model(alpha=alpha).fit(X_tr, y_train)
          f1 = f1_score(y_test, model.predict(X_te), average='macro')
          resultados.append((f1, vect_name, stop_words, min_df, model_name, alpha))

resultados.sort(reverse=True)

Mostramos las 10 mejores configuraciones:

In [168]:
for f1, vect_name, stop_words, min_df, model_name, alpha in resultados[:10]:
  print(f'F1={f1:.4f} | {vect_name:5s} | stop_words={str(stop_words):7s} | min_df={min_df} | {model_name:13s} | alpha={alpha}')

F1=0.6999 | tfidf | stop_words=None    | min_df=1 | ComplementNB  | alpha=0.3
F1=0.6993 | tfidf | stop_words=english | min_df=1 | ComplementNB  | alpha=0.3
F1=0.6976 | tfidf | stop_words=None    | min_df=2 | ComplementNB  | alpha=0.3
F1=0.6958 | tfidf | stop_words=english | min_df=2 | ComplementNB  | alpha=0.3
F1=0.6954 | tfidf | stop_words=None    | min_df=1 | ComplementNB  | alpha=0.1
F1=0.6943 | tfidf | stop_words=english | min_df=2 | ComplementNB  | alpha=1.0
F1=0.6936 | tfidf | stop_words=english | min_df=1 | ComplementNB  | alpha=1.0
F1=0.6935 | tfidf | stop_words=None    | min_df=2 | ComplementNB  | alpha=1.0
F1=0.6930 | tfidf | stop_words=None    | min_df=1 | ComplementNB  | alpha=1.0
F1=0.6919 | tfidf | stop_words=english | min_df=1 | ComplementNB  | alpha=0.1


Y la mejor configuración de cada modelo, para compararlos:

In [169]:
for model_name in ['MultinomialNB', 'ComplementNB']:
  mejor = next(r for r in resultados if r[4] == model_name)
  print(f'{model_name}: F1={mejor[0]:.4f} | {mejor[1]} | stop_words={mejor[2]} | min_df={mejor[3]} | alpha={mejor[5]}')

MultinomialNB: F1=0.6844 | tfidf | stop_words=english | min_df=1 | alpha=0.01
ComplementNB: F1=0.6999 | tfidf | stop_words=None | min_df=1 | alpha=0.3


**Interpretación:** Los mejores resultados se obtienen con TF-IDF y `ComplementNB` con `alpha=0.3` (F1 macro 0.700), por encima del mejor `MultinomialNB` (0.684, con `alpha=0.01`). Con la configuración inicial (`MultinomialNB`, `alpha=1.0`) se obtenía 0.585, por lo que el mayor aporte vino de reducir `alpha`: con `alpha=1.0` el suavizado de Laplace es demasiado fuerte con vocabularios de ~100 mil términos y pocas apariciones por palabra.

`ComplementNB` rinde mejor que `MultinomialNB` en este dataset, lo cual es esperable porque estima los parámetros con los documentos de las demás clases y es más robusto cuando las clases están algo desbalanceadas. Los vectorizadores con TF-IDF superan a los de conteos (ninguna configuración de conteos aparece entre las 10 mejores). Sacar stop words o subir `min_df` casi no cambia el resultado (diferencias menores a 0.005): las stop words ya pesan poco con TF-IDF. Todos los Naïve Bayes ajustados superan ampliamente al clasificador por similaridad del ejercicio 2 (0.505).

## Ejercicio 4: Similaridad entre palabras

Transponemos la matriz documento-término para obtener la matriz término-documento. Ahora cada fila es el vector de una palabra, que representa en qué documentos aparece y con qué peso TF-IDF.

In [170]:
X_train_T = X_train.T
print(f'shape: {X_train_T.shape}')

shape: (101631, 11314)


Elegimos manualmente 5 palabras de temas distintos, para que los resultados sean interpretables:

In [171]:
palabras = ['car', 'god', 'space', 'hockey', 'computer']

for palabra in palabras:
  print(palabra, tfidfvect.vocabulary_[palabra])

car 25775
god 43842
space 84097
hockey 47021
computer 28940


Para cada palabra medimos la similaridad coseno con todas las demás y mostramos las 5 más similares (salteando la primera, que es la palabra misma).

In [172]:
for palabra in palabras:
  idx_palabra = tfidfvect.vocabulary_[palabra]
  cossim_palabra = cosine_similarity(X_train_T[idx_palabra], X_train_T)[0]
  mostsim_palabra = np.argsort(cossim_palabra)[::-1][1:6]

  print(f'{palabra}:')
  for i in mostsim_palabra:
    print(f'  {idx2word[i]:15s} sim={cossim_palabra[i]:.3f}')

car:
  cars            sim=0.180
  criterium       sim=0.177
  civic           sim=0.175
  owner           sim=0.169
  dealer          sim=0.168
god:
  jesus           sim=0.269
  bible           sim=0.262
  that            sim=0.256
  existence       sim=0.255
  christ          sim=0.251
space:
  nasa            sim=0.330
  seds            sim=0.297
  shuttle         sim=0.293
  enfant          sim=0.280
  seti            sim=0.246
hockey:
  ncaa            sim=0.274
  nhl             sim=0.265
  affiliates      sim=0.248
  xenophobes      sim=0.243
  sportschannel   sim=0.223
computer:
  decwriter       sim=0.156
  deluged         sim=0.152
  harkens         sim=0.152
  shopper         sim=0.144
  the             sim=0.136


**Interpretación:** Con la matriz término-documento, dos palabras son similares si aparecen en los mismos documentos, por lo que la similaridad refleja co-ocurrencia temática.
* **car:** `cars`, `civic`, `owner`, `dealer`: vocabulario de autos y compra-venta. `criterium` es un caso curioso (probablemente aparece en pocos documentos del mismo tema).
* **god:** `jesus`, `bible`, `christ`, `existence`: vocabulario religioso y de debates sobre la existencia de Dios. Aparece también `that`, una stop word, ya que `god` es muy frecuente en posts largos.
* **space:** `nasa`, `shuttle`, `seti`, `seds`, `enfant`: espacio y astronomía (los dos últimos son siglas y nombres de la comunidad de sci.space).
* **hockey:** `nhl`, `ncaa`, `sportschannel`, `affiliates`: deportes y ligas. `xenophobes` es llamativa, pero probablemente proviene de un hilo puntual de la clase hockey.
* **computer:** es la menos interpretable (`decwriter`, `deluged`, `harkens`, `shopper`, `the`): al ser una palabra tan general aparece en documentos de muchos temas, y los vecinos son términos raros que comparten unos pocos documentos.

Se observa que las palabras con tema muy definido (`space`, `god`, `car`) tienen vecinos muy coherentes, mientras que las palabras generales no. Además, las palabras raras con pocos documentos pueden tener similaridad alta con una palabra solo porque comparten uno o dos documentos.